# Сравнение DLightSB и DLightSB-MM

Ноутбук принимает один или два ZIP-архива созданных ноутбуком `run_dlight_sb_pairs_colab.ipynb`

Можно анализировать одну модель отдельно или сравнить сохранённый baseline с новым MM-запуском

Старые парные архивы также поддерживаются


## Импорты и оформление


In [ ]:
from pathlib import Path
import json
import shutil
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

plt.style.use("default")
plt.rcParams.update(
    {
        "figure.figsize": (9, 5),
        "axes.grid": True,
        "grid.alpha": 0.25,
        "lines.linewidth": 2,
        "lines.markersize": 6,
    }
)


## Загрузка архивов

Выберите один архив для отдельного анализа или два архива для сравнения baseline и MM


In [ ]:
from google.colab import files

uploaded = files.upload()
archive_names = [
    name
    for name in uploaded
    if name.lower().endswith(".zip")
]

if not 1 <= len(archive_names) <= 2:
    raise ValueError(
        f"Нужно загрузить один или два ZIP-архива получено {len(archive_names)}"
    )

ARCHIVE_PATHS = [Path(name).resolve() for name in sorted(archive_names)]
print("Загружены архивы:")
for path in ARCHIVE_PATHS:
    print("-", path.name)


## Распаковка и чтение результатов


In [ ]:
ANALYSIS_ROOT = Path("/content/catsbench_model_analysis")

if ANALYSIS_ROOT.is_dir():
    shutil.rmtree(ANALYSIS_ROOT)

ANALYSIS_ROOT.mkdir(parents=True)


def extract_archive(archive_path: Path, extract_dir: Path) -> None:
    extract_dir.mkdir(parents=True)
    root = extract_dir.resolve()

    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (extract_dir / member.filename).resolve()

            if not target.is_relative_to(root):
                raise ValueError(
                    f"Некорректный путь внутри архива: {member.filename}"
                )

        archive.extractall(extract_dir)


EXTRACTED_ARCHIVES = []

for index, archive_path in enumerate(ARCHIVE_PATHS):
    extract_dir = ANALYSIS_ROOT / f"{index}_{archive_path.stem}"
    extract_archive(archive_path, extract_dir)
    EXTRACTED_ARCHIVES.append((archive_path, extract_dir))
    print(f"{archive_path.name} распакован в {extract_dir}")


In [ ]:
MODEL_LABELS = {
    "baseline": "Gradient descent",
    "mm": "MM",
}

MODEL_COLORS = {
    "Gradient descent": "#1f77b4",
    "MM": "#ff7f0e",
}


def find_one(run_dir: Path, pattern: str) -> Path:
    matches = sorted(run_dir.glob(pattern))

    if len(matches) != 1:
        raise FileNotFoundError(
            f"Для {pattern} ожидался один файл найдено {len(matches)}"
        )

    return matches[0]


def find_logger_file(run_dir: Path, logger_name: str, relative_path: str) -> Path:
    preferred = run_dir / logger_name / relative_path

    if preferred.is_file():
        return preferred

    return find_one(run_dir, f"*/{relative_path}")


def read_run(run_dir: Path) -> dict:
    train_metrics_path = find_logger_file(run_dir, "csv", "train/metrics.csv")
    test_metrics_path = find_logger_file(run_dir, "csv", "test/metrics.csv")
    hparams_path = find_logger_file(run_dir, "csv", "train/hparams.yaml")

    with hparams_path.open(encoding="utf-8") as file:
        config = yaml.safe_load(file)

    return {
        "run_dir": run_dir,
        "train": pd.read_csv(train_metrics_path),
        "test": pd.read_csv(test_metrics_path),
        "config": config,
    }


def archived_models(extract_dir: Path, manifest: dict) -> list[tuple[str, Path]]:
    if manifest.get("archive_format") == "single_run_v1":
        kind = manifest["kind"]
        return [(kind, extract_dir / kind)]

    if "experiments" in manifest:
        return [
            (kind, extract_dir / kind)
            for kind in MODEL_LABELS
            if kind in manifest["experiments"]
        ]

    raise ValueError("Неизвестный формат архива")


runs = {}
metadata_rows = {}

for archive_path, extract_dir in EXTRACTED_ARCHIVES:
    with (extract_dir / "manifest.json").open(encoding="utf-8") as file:
        manifest = json.load(file)

    for kind, run_dir in archived_models(extract_dir, manifest):
        if kind not in MODEL_LABELS:
            raise ValueError(f"Неизвестный тип модели: {kind}")

        label = MODEL_LABELS[kind]

        if label in runs:
            raise ValueError(f"Загружено несколько архивов модели {label}")

        runs[label] = read_run(run_dir)
        metadata_rows[label] = {
            "dimension": int(manifest["dimension"]),
            "seed": int(manifest["seed"]),
            "commit": manifest["commit"],
            "archive": archive_path.name,
        }

if not runs:
    raise RuntimeError("В архивах не найдено ни одного запуска")

dimensions = {row["dimension"] for row in metadata_rows.values()}

if len(dimensions) != 1:
    raise ValueError(f"Размерности запусков не совпадают: {sorted(dimensions)}")

DIMENSION = dimensions.pop()
metadata = pd.DataFrame.from_dict(metadata_rows, orient="index")
display(metadata.style.set_caption("Загруженные запуски"))


## Финальные метрики

Для score и effective k лучшим считается большее значение

Для loss и log c лучшим считается меньшее значение

Для log v лучшим считается большее значение поскольку `loss = log_c - log_v`


In [ ]:
SUMMARY_COLUMNS = [
    "cond_shape_score_forward",
    "cond_trend_score_forward",
    "loss",
    "log_c",
    "log_v",
    "posterior_eff_k",
    "prior_eff_k",
    "shape_score_forward",
    "trend_score_forward",
]

MINIMIZE_COLUMNS = {"loss", "log_c"}
BEST_COLOR = "#6f42c1"


def final_test_values(frame: pd.DataFrame) -> dict:
    values = {}

    for column in frame.columns:
        if not column.startswith("test/"):
            continue

        series = frame[column].dropna()

        if len(series):
            values[column.removeprefix("test/")] = float(series.iloc[-1])

    if "log_c" not in values or "log_v" not in values:
        raise KeyError("В test CSV отсутствуют log_c или log_v")

    values["loss"] = values["log_c"] - values["log_v"]
    return values


def highlight_best(frame: pd.DataFrame) -> pd.DataFrame:
    styles = pd.DataFrame("", index=frame.index, columns=frame.columns)

    if len(frame.index) < 2:
        return styles

    for column in frame.columns:
        valid = frame[column].dropna()

        if valid.empty:
            continue

        best = valid.min() if column in MINIMIZE_COLUMNS else valid.max()
        winners = np.isclose(frame[column], best, rtol=1e-9, atol=1e-12)
        styles.loc[winners, column] = (
            f"background-color: {BEST_COLOR}; color: white; font-weight: bold"
        )

    return styles


In [ ]:
summary = pd.DataFrame(
    {
        label: final_test_values(run["test"])
        for label, run in runs.items()
    }
).T

missing_columns = [
    column
    for column in SUMMARY_COLUMNS
    if column not in summary.columns
]

if missing_columns:
    raise KeyError(
        "В итоговых логах отсутствуют метрики: "
        + ", ".join(missing_columns)
    )

summary = summary[SUMMARY_COLUMNS]

display(
    summary.style
    .format(precision=5, na_rep="—")
    .apply(highlight_best, axis=None)
    .set_caption("Финальные test-метрики")
)


## Train loss по всем доступным записям

Для Gradient descent loss рассчитывается по каждой паре `train/log_c` и `train/log_v`

Для MM используется непосредственно `train/loss`

Каждая точка соответствует реально сохранённой train-записи без интерполяции между логами


In [ ]:
def last_non_null(series: pd.Series):
    values = series.dropna()
    return values.iloc[-1] if len(values) else np.nan


def training_progress(run: dict) -> pd.DataFrame:
    frame = run["train"]
    columns = [
        "train/time",
        "train/batches",
        "train/samples",
        "train/device/samples_per_sec",
    ]
    missing = [
        column
        for column in ["step", *columns[:3]]
        if column not in frame.columns
    ]

    if missing:
        raise KeyError(
            "В train CSV отсутствуют метрики ThroughputMonitor: "
            + ", ".join(missing)
        )

    available = [column for column in columns if column in frame.columns]
    progress = (
        frame[["step", *available]]
        .groupby("step", as_index=False)
        .agg({column: last_non_null for column in available})
        .dropna(subset=["train/time", "train/samples"])
        .sort_values("train/samples")
        .reset_index(drop=True)
    )
    return progress


def training_loss_history(run: dict) -> pd.DataFrame:
    frame = run["train"]
    progress_columns = [
        "train/time",
        "train/batches",
        "train/samples",
    ]

    if "train/loss" in frame.columns:
        loss_columns = ["train/loss"]
    elif {"train/log_c", "train/log_v"}.issubset(frame.columns):
        loss_columns = ["train/log_c", "train/log_v"]
    else:
        raise KeyError(
            "В train CSV нет ни train/loss ни пары train/log_c и train/log_v"
        )

    required = ["step", *progress_columns, *loss_columns]
    missing = [column for column in required if column not in frame.columns]

    if missing:
        raise KeyError(
            "В train CSV отсутствуют колонки: " + ", ".join(missing)
        )

    value_columns = [*progress_columns, *loss_columns]
    history = (
        frame[["step", *value_columns]]
        .groupby("step", as_index=False)
        .agg({column: last_non_null for column in value_columns})
        .dropna(subset=[*progress_columns, *loss_columns])
        .sort_values("train/samples")
        .reset_index(drop=True)
    )

    if loss_columns == ["train/loss"]:
        history["loss"] = history["train/loss"]
    else:
        history["loss"] = history["train/log_c"] - history["train/log_v"]

    return history.rename(
        columns={
            "train/time": "train_time_seconds",
            "train/batches": "batches_seen",
            "train/samples": "samples_seen",
        }
    )


In [ ]:
loss_histories = {
    label: training_loss_history(run)
    for label, run in runs.items()
}

for label, history in loss_histories.items():
    print(
        label,
        f"loss points={len(history)}",
        f"batches={int(history['batches_seen'].min()):,}..{int(history['batches_seen'].max()):,}",
        f"samples={int(history['samples_seen'].min()):,}..{int(history['samples_seen'].max()):,}",
    )


In [ ]:
fig, axis = plt.subplots(figsize=(10, 5))

for label, history in loss_histories.items():
    axis.plot(
        history["samples_seen"],
        history["loss"],
        marker=".",
        markersize=3,
        linewidth=1.3,
        color=MODEL_COLORS[label],
        label=f"{label} {len(history)} точек",
    )

axis.set_xscale("log")
axis.set_xlabel("Обработано объектов")
axis.set_ylabel("Train loss")
axis.set_title(f"Train loss по всем записям d={DIMENSION}")
axis.legend()
plt.tight_layout()
plt.show()


## Условные validation-метрики по объёму обработанных данных

Эти метрики существуют только в моменты validation


In [ ]:
VALIDATION_COLUMNS = [
    "val/cond_shape_score_forward",
    "val/cond_trend_score_forward",
]


def validation_history(run: dict) -> pd.DataFrame:
    frame = run["train"]
    missing = [
        column
        for column in ["step", *VALIDATION_COLUMNS]
        if column not in frame.columns
    ]

    if missing:
        raise KeyError(
            "В train CSV отсутствуют validation-колонки: "
            + ", ".join(missing)
        )

    history = (
        frame[["step", *VALIDATION_COLUMNS]]
        .dropna(subset=VALIDATION_COLUMNS, how="all")
        .groupby("step", as_index=False)
        .agg({column: last_non_null for column in VALIDATION_COLUMNS})
        .sort_values("step")
        .reset_index(drop=True)
    )

    batch_size = int(run["config"]["data"]["batch_size"])
    accumulate = int(
        run["config"].get("trainer", {}).get("accumulate_grad_batches", 1)
    )
    history["expected_samples"] = (
        (history["step"].astype(int) + 1)
        * batch_size
        * accumulate
    ).astype(float)

    progress = training_progress(run).copy()
    progress["train/samples"] = progress["train/samples"].astype(float)
    history = pd.merge_asof(
        history.sort_values("expected_samples"),
        progress,
        left_on="expected_samples",
        right_on="train/samples",
        direction="nearest",
    )
    history["samples_seen"] = history["train/samples"].astype(int)
    history["train_time_seconds"] = history["train/time"]
    return history


In [ ]:
histories = {
    label: validation_history(run)
    for label, run in runs.items()
}

for label, history in histories.items():
    print(
        label,
        f"validation points={len(history)}",
        f"samples={history['samples_seen'].min():,}..{history['samples_seen'].max():,}",
    )


In [ ]:
validation_plots = [
    ("val/cond_shape_score_forward", "Conditional shape score"),
    ("val/cond_trend_score_forward", "Conditional trend score"),
]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for axis, (column, title) in zip(axes, validation_plots):
    for label, history in histories.items():
        points = history.dropna(subset=[column])
        axis.plot(
            points["samples_seen"],
            points[column],
            marker="o",
            color=MODEL_COLORS[label],
            label=label,
        )

    axis.set_xscale("log")
    axis.set_title(title)
    axis.set_xlabel("Обработано объектов")
    axis.legend()

axes[0].set_ylabel("Значение метрики")
fig.suptitle(f"Validation-метрики d={DIMENSION}", fontsize=14)
plt.tight_layout()
plt.show()


## Train loss по чистому времени обучения

Используются те же полные train-записи loss


In [ ]:
fig, axis = plt.subplots(figsize=(10, 5))

for label, history in loss_histories.items():
    axis.plot(
        history["train_time_seconds"],
        history["loss"],
        marker=".",
        markersize=3,
        linewidth=1.3,
        color=MODEL_COLORS[label],
        label=f"{label} {len(history)} точек",
    )

axis.set_xlabel("Чистое время обучения секунд")
axis.set_ylabel("Train loss")
axis.set_title(f"Train loss по времени d={DIMENSION}")
axis.legend()
plt.tight_layout()
plt.show()


## Условные validation-метрики по чистому времени обучения

`train/time` формируется callback `ThroughputMonitor` и не включает время validation


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for axis, (column, title) in zip(axes, validation_plots):
    for label, history in histories.items():
        points = history.dropna(subset=[column, "train_time_seconds"])
        axis.plot(
            points["train_time_seconds"],
            points[column],
            marker="o",
            color=MODEL_COLORS[label],
            label=label,
        )

    axis.set_title(title)
    axis.set_xlabel("Чистое время обучения секунд")
    axis.legend()

axes[0].set_ylabel("Значение метрики")
fig.suptitle(f"Validation-метрики по времени d={DIMENSION}", fontsize=14)
plt.tight_layout()
plt.show()


## Чистое время обучения по объёму обработанных данных


In [ ]:
progress = {
    label: training_progress(run)
    for label, run in runs.items()
}

fig, axis = plt.subplots(figsize=(9, 5))

for label, frame in progress.items():
    axis.plot(
        frame["train/samples"],
        frame["train/time"],
        marker="o",
        color=MODEL_COLORS[label],
        label=label,
    )

axis.set_xscale("log")
axis.set_xlabel("Обработано объектов")
axis.set_ylabel("Чистое время обучения секунд")
axis.set_title("Время обучения по объёму данных")
axis.legend()
plt.tight_layout()
plt.show()


## Скорость обучения

Среднее время итерации и throughput рассчитаны из накопленного train-времени без validation


In [ ]:
performance_rows = {}

for label, frame in progress.items():
    final = frame.iloc[-1]
    seconds = float(final["train/time"])
    batches = int(final["train/batches"])
    samples = int(final["train/samples"])

    performance_rows[label] = {
        "train_time_seconds": seconds,
        "iterations": batches,
        "samples": samples,
        "seconds_per_iteration": seconds / batches,
        "samples_per_second": samples / seconds,
        "samples_per_minute": 60 * samples / seconds,
    }

performance = pd.DataFrame.from_dict(performance_rows, orient="index")
display(performance.style.format(precision=3).set_caption("Скорость обучения"))


## Использование памяти GPU во время обучения

Показан накопленный пик памяти PyTorch CUDA allocator после train-update

Allocated peak соответствует максимальной памяти занятой тензорами с начала процесса

Reserved peak дополнительно включает максимальный размер кэша CUDA allocator


In [ ]:
GPU_ALLOCATED_COLUMN = (
    "DeviceStatsMonitor.on_train_batch_end/allocated_bytes.all.peak"
)
GPU_RESERVED_COLUMN = (
    "DeviceStatsMonitor.on_train_batch_end/reserved_bytes.all.peak"
)


def gpu_memory_history(run: dict) -> pd.DataFrame:
    frame = run["train"]
    required = ["step", GPU_ALLOCATED_COLUMN, GPU_RESERVED_COLUMN]
    missing = [column for column in required if column not in frame.columns]

    if missing:
        raise KeyError(
            "В train CSV отсутствуют GPU memory-метрики: "
            + ", ".join(missing)
        )

    memory = (
        frame[required]
        .dropna(subset=[GPU_ALLOCATED_COLUMN, GPU_RESERVED_COLUMN], how="all")
        .groupby("step", as_index=False)
        .agg(
            {
                GPU_ALLOCATED_COLUMN: last_non_null,
                GPU_RESERVED_COLUMN: last_non_null,
            }
        )
        .sort_values("step")
    )
    train_progress = training_progress(run).sort_values("step")
    memory = pd.merge_asof(
        memory,
        train_progress[["step", "train/samples"]],
        on="step",
        direction="nearest",
    )
    memory["allocated_mib"] = memory[GPU_ALLOCATED_COLUMN] / 1024**2
    memory["reserved_mib"] = memory[GPU_RESERVED_COLUMN] / 1024**2
    return memory


In [ ]:
memory_histories = {
    label: gpu_memory_history(run)
    for label, run in runs.items()
}

model_count = len(memory_histories)
fig, axes = plt.subplots(
    1,
    model_count,
    figsize=(7 * model_count, 5),
    squeeze=False,
)
axes = axes.ravel()

for axis, (label, frame) in zip(axes, memory_histories.items()):
    axis.plot(
        frame["train/samples"],
        frame["allocated_mib"],
        label="Allocated peak",
        color=MODEL_COLORS[label],
    )
    axis.plot(
        frame["train/samples"],
        frame["reserved_mib"],
        label="Reserved peak",
        color=MODEL_COLORS[label],
        linestyle="--",
        alpha=0.8,
    )
    axis.set_xscale("log")
    axis.set_title(label)
    axis.set_xlabel("Обработано объектов")
    axis.set_ylabel("GPU memory MiB")
    axis.legend()

fig.suptitle("Пиковая память GPU во время обучения", fontsize=14)
plt.tight_layout()
plt.show()
